In [16]:
import pandas as pd
import numpy as np

# Cleaning and Filtering the Dataset

In [30]:
df = pd.read_csv("../butterflies/data/butterflies/metadata.csv")

In [31]:
df.head()

,image_id,filename,raw_filename,species,common_name,family,subfamily,genus,order,life_stage,sex,media_code,location,state,date,credit,source_url,source,split
0,1,butterflies/images/Abisara_bifasciata/Abisara_...,butterflies/raw/Abisara_bifasciata/Abisara_bif...,NaN,NaN,NaN,NaN,NaN,Lepidoptera,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ifoundbutterflies.org,test
1,2,butterflies/images/Abisara_fylla/Abisara_fylla...,butterflies/raw/Abisara_fylla/Abisara_fylla_Ad...,NaN,NaN,NaN,NaN,NaN,Lepidoptera,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ifoundbutterflies.org,train
2,3,butterflies/images/Acraea_terpsicore/Acraea_te...,butterflies/raw/Acraea_terpsicore/Acraea_terps...,Acraea terpsicore,Tawny Coster,Nymphalidae,Heliconiinae,Acraea,Lepidoptera,Early Stage,NaN,NaN,NaN,NaN,NaN,NaN,https://www.ifoundbutterflies.org/Acraea-terps...,ifoundbutterflies.org,val
3,4,butterflies/images/Araotes_lapithis/Araotes_la...,butterflies/raw/Araotes_lapithis/Araotes_lapit...,Araotes lapithis,Witch,Lycaenidae,Theclinae,Araotes,Lepidoptera,Adult/Unknown,NaN,NaN,NaN,NaN,NaN,NaN,https://www.ifoundbutterflies.org/Araotes-lapi...,ifoundbutterflies.org,train
4,5,butterflies/images/Argynnis_childreni/Argynnis...,butterflies/raw/Argynnis_childreni/Argynnis_ch...,Argynnis childreni,Large Silverstripe,Nymphalidae,Heliconiinae,Argynnis,Lepidoptera,Adult/Unknown,NaN,NaN,NaN,NaN,NaN,NaN,https://www.ifoundbutterflies.org/Argynnis-chi...,ifoundbutterflies.org,test


In [32]:
df.shape

(60641, 19)

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60641 entries, 0 to 60640
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   image_id      60641 non-null  int64 
 1   filename      60641 non-null  object
 2   raw_filename  60641 non-null  object
 3   species       60637 non-null  object
 4   common_name   60380 non-null  object
 5   family        60490 non-null  object
 6   subfamily     60490 non-null  object
 7   genus         60490 non-null  object
 8   order         60641 non-null  object
 9   life_stage    60490 non-null  object
 10  sex           21332 non-null  object
 11  media_code    55766 non-null  object
 12  location      60542 non-null  object
 13  state         56712 non-null  object
 14  date          46216 non-null  object
 15  credit        43461 non-null  object
 16  source_url    60490 non-null  object
 17  source        60641 non-null  object
 18  split         60641 non-null  object
dtypes: i

In [34]:
columns_to_drop = ["raw_filename", "sex", "media_code", "date", "credit", "source_url"]
df = df.drop(columns=columns_to_drop)

In [35]:
df.isnull().sum()

image_id          0
filename          0
species           4
common_name     261
family          151
subfamily       151
genus           151
order             0
life_stage      151
location         99
state          3929
source            0
split             0
dtype: int64

In [36]:
df = df[df["species"].notna()].copy()
print(df.shape)

(60637, 13)


In [37]:
mask = df[["family", "subfamily", "genus", "life_stage"]].isnull().any(axis=1)
print(mask.sum())

147


In [38]:
df = df.dropna(subset=["family", "subfamily", "genus", "life_stage"])
print(df.shape)

(60490, 13)


In [39]:
df["state"] = df["state"].fillna("Unknown")

In [40]:
df.isnull().sum()

image_id         0
filename         0
species          0
common_name    257
family           0
subfamily        0
genus            0
order            0
life_stage       0
location        93
state            0
source           0
split            0
dtype: int64

In [41]:
df = df.dropna(subset=['location'])

In [42]:
df["common_name"] = df["common_name"].fillna(df["species"])

In [43]:
df.isnull().sum()

image_id       0
filename       0
species        0
common_name    0
family         0
subfamily      0
genus          0
order          0
life_stage     0
location       0
state          0
source         0
split          0
dtype: int64

In [45]:
needed_species = [
    "Anthene lycaenina", "Azanus jesous", "Azanus ubaldus", "Azanus uranus",
    "Badamia exclamationis", "Catochrysops strabo", "Colias erate", "Colias fieldii",
    "Danaus chrysippus", "Danaus genutia", "Delias eucharis", "Euchrysops cnejus",
    "Euploea core", "Eurema hecabe", "Everes argiades", "Freyeria putli",
    "Freyeria trochylus", "Graphium agamemnon", "Graphium doson", "Hasora badra",
    "Jamides celeno", "Junonia hierta", "Junonia orithya", "Lachides parrhasius",
    "Lacturnea lacturnus", "Lampides boeticus", "Leptotes plinius", "Luthrodes pandava",
    "Neptis hylas", "Parantica aglea", "Prosotas dubiosa", "Prosotas nora",
    "Rapala iarbus", "Tirumala limniace", "Zizina otis",
]

df_final = df[df["species"].isin(needed_species)].copy()
print("Species matched:", df_final["species"].nunique())
print("Images:", len(df_final))

Species matched: 33
Images: 4570


In [47]:
split_balance = df_final.groupby(["species", "split"]).size().unstack(fill_value=0)
print(split_balance)

# quick check for species with zero images in val or test
no_val = split_balance[split_balance.get("val", 0) == 0]
no_test = split_balance[split_balance.get("test", 0) == 0]
print(f"\nSpecies with 0 val images: {len(no_val)}")
print(f"Species with 0 test images: {len(no_test)}")

split                  test  train  val
species                                
Anthene lycaenina        17    126   15
Azanus jesous            15    113   14
Azanus ubaldus           16    128   16
Azanus uranus            12     92   11
Badamia exclamationis    15    112   14
Catochrysops strabo      15    113   14
Colias erate              6     36    4
Colias fieldii           15    109   13
Danaus chrysippus        16    122   15
Danaus genutia           15    110   13
Delias eucharis          14    107   13
Euchrysops cnejus        16    123   15
Euploea core             15    110   13
Eurema hecabe            14    104   13
Everes argiades           5     30    3
Freyeria putli           14    107   13
Freyeria trochylus       13     97   12
Graphium agamemnon       15    108   13
Graphium doson           15    107   13
Hasora badra             18    134   16
Jamides celeno           15    110   13
Junonia hierta           16    121   15
Lachides parrhasius      16    123   15


In [48]:
import os
image_root = "../butterflies/data"

In [49]:
df_final["file_exists"] = df_final["filename"].apply(
    lambda f: os.path.exists(os.path.join(image_root, f))
)
print(df_final["file_exists"].value_counts())

file_exists
True    4570
Name: count, dtype: int64


In [52]:
from PIL import Image

def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False

image_root = "../butterflies/data"

df_final["file_valid"] = df_final["filename"].apply(
    lambda f: is_valid_image(os.path.join(image_root, f))
)
print(df_final["file_valid"].value_counts())

file_valid
True    4570
Name: count, dtype: int64


In [53]:
df_final_to_save = df_final.drop(columns=["file_exists", "file_valid"], errors="ignore")
df_final_to_save.to_csv("butterflies_metadata_clean.csv", index=False)

In [54]:
df = pd.read_csv("../plants/data/plants/metadata.csv")

In [55]:
df.head()

,image_id,filename,raw_filename,species,family,genus,image_type,media_code,location,state,date,credit,butterfly_hosts,source_url,source,split,sex
0,1,plants/images/Abelmoschus_esculentus/Abelmosch...,plants/raw/Abelmoschus_esculentus/Abelmoschus_...,Abelmoschus esculentus,Malvaceae,NaN,gallery,bz705,Tarapur; Boisar; Palghar District; Maharshtra;...,NaN,2007/09/15,Paresh Churi,NaN,NaN,ifoundbutterflies.org,train,NaN
1,2,plants/images/Abelmoschus_esculentus/Abelmosch...,plants/raw/Abelmoschus_esculentus/Abelmoschus_...,Abelmoschus esculentus,Malvaceae,NaN,hero,bz760,"Mahim, Maharashtra Nature Park, Mumbai Distric...",Maharashtra,2008/08/18,Paresh Churi,NaN,NaN,ifoundbutterflies.org,test,NaN
2,3,plants/images/Abrus_precatorius/Abrus_precator...,plants/raw/Abrus_precatorius/Abrus_precatorius...,Abrus precatorius,precatorius,NaN,gallery,BOI-ppw838,"Mithbav; Devgad Taluka, Sindhudurg District, M...",Maharashtra,NaN,NaN,NaN,NaN,ifoundbutterflies.org,val,NaN
3,4,plants/images/Abrus_precatorius/Abrus_precator...,plants/raw/Abrus_precatorius/Abrus_precatorius...,Abrus precatorius,Fabaceae,NaN,gallery,9g676,"CEC BNHS, Goregaon; Mumbai Suburban District, ...",Maharashtra,2021/09/04,Raju Kasambe,NaN,NaN,ifoundbutterflies.org,train,NaN
4,5,plants/images/Abrus_precatorius/Abrus_precator...,plants/raw/Abrus_precatorius/Abrus_precatorius...,Abrus precatorius,Fabaceae,NaN,gallery,gd273,"CEC BNHS, Goregaon; Mumbai Suburban District, ...",Maharashtra,2019/08/29,Raju Kasambe,NaN,NaN,ifoundbutterflies.org,train,NaN


In [56]:
df.shape

(703, 17)

In [57]:
df.isnull().sum()

image_id             0
filename             0
raw_filename         0
species              0
family               0
genus              559
image_type           0
media_code          30
location             0
state               12
date                85
credit             136
butterfly_hosts    560
source_url         559
source               0
split                0
sex                703
dtype: int64

In [58]:
columns_to_drop = ["raw_filename", "genus", "media_code", "date", "credit", "source_url", "sex"]
df = df.drop(columns=columns_to_drop)

In [59]:
df_filtered = df[df["butterfly_hosts"].notna()].copy()
print(df_filtered.shape)
print(df_filtered["species"].nunique())

(143, 10)
19


In [60]:
species_counts = df_filtered.groupby("species").size()

In [61]:
species_counts.describe()

count    19.000000
mean      7.526316
std       4.914473
min       2.000000
25%       3.500000
50%       7.000000
75%       9.500000
max      18.000000
dtype: float64

In [62]:
valid_species = species_counts[species_counts >= 4].index
df_final = df_filtered[df_filtered["species"].isin(valid_species)].copy()

print("Species remaining:", df_final["species"].nunique())
print("Total images remaining:", len(df_final))

Species remaining: 14
Total images remaining: 132


In [63]:
split_balance = df_final.groupby(["species", "split"]).size().unstack(fill_value=0)
print(split_balance)

no_val = split_balance[split_balance.get("val", 0) == 0]
no_test = split_balance[split_balance.get("test", 0) == 0]
print(f"Species with 0 val images: {len(no_val)}")
print(f"Species with 0 test images: {len(no_test)}")

split                     test  train  val
species                                   
Alysicarpus vaginalis        2      6    0
Gomphocarpus physocarpus     2      5    0
Hemidesmus indicus           1      3    0
Libidibia coriaria           2     10    1
Lotus corniculatus           2      4    0
Macroptilium lathyroides     1      4    0
Mimosa hamata                2      6    0
Mimosa pudica                2     11    1
Trigonella emodi             1      8    1
Vachellia farnesiana         1      4    0
Vachellia leucophloea        2      7    0
Vachellia nilotica           3     14    1
Vigna radiata                3     12    1
Vigna unguiculata            2      7    0
Species with 0 val images: 9
Species with 0 test images: 0


In [64]:
species_needing_val = split_balance[split_balance.get("val", 0) == 0].index.tolist()
print(species_needing_val)

['Alysicarpus vaginalis', 'Gomphocarpus physocarpus', 'Hemidesmus indicus', 'Lotus corniculatus', 'Macroptilium lathyroides', 'Mimosa hamata', 'Vachellia farnesiana', 'Vachellia leucophloea', 'Vigna unguiculata']


In [65]:
np.random.seed(42)
for sp in species_needing_val:
    train_rows = df_final[(df_final["species"] == sp) & (df_final["split"] == "train")]
    if len(train_rows) > 1:  # keep at least 1 image in train
        move_idx = train_rows.sample(1, random_state=42).index
        df_final.loc[move_idx, "split"] = "val"

In [66]:
split_balance_fixed = df_final.groupby(["species", "split"]).size().unstack(fill_value=0)
print(split_balance_fixed)

no_val = split_balance_fixed[split_balance_fixed.get("val", 0) == 0]
print(f"Species with 0 val images: {len(no_val)}")

split                     test  train  val
species                                   
Alysicarpus vaginalis        2      5    1
Gomphocarpus physocarpus     2      4    1
Hemidesmus indicus           1      2    1
Libidibia coriaria           2     10    1
Lotus corniculatus           2      3    1
Macroptilium lathyroides     1      3    1
Mimosa hamata                2      5    1
Mimosa pudica                2     11    1
Trigonella emodi             1      8    1
Vachellia farnesiana         1      3    1
Vachellia leucophloea        2      6    1
Vachellia nilotica           3     14    1
Vigna radiata                3     12    1
Vigna unguiculata            2      6    1
Species with 0 val images: 0


In [67]:
len(df_final)

132

In [68]:
print(df_final["butterfly_hosts"].head(10).tolist())

['Zizina otis', 'Zizina otis', 'Zizina otis', 'Zizina otis', 'Zizina otis', 'Zizina otis', 'Zizina otis', 'Zizina otis', 'Danaus chrysippus, Danaus chrysippus', 'Danaus chrysippus, Danaus chrysippus']


In [69]:
df_final["butterfly_hosts_list"] = (
    df_final["butterfly_hosts"]
    .str.split(",")
    .apply(lambda lst: [name.strip() for name in lst])
)

In [70]:
df_final["num_hosts"] = df_final["butterfly_hosts_list"].apply(len)
print(df_final["num_hosts"].value_counts().sort_index())

num_hosts
1     12
2     35
4     39
6     15
7     18
15    13
Name: count, dtype: int64


In [71]:
df_hosts_long = df_final.explode("butterfly_hosts_list").rename(
    columns={"butterfly_hosts_list": "butterfly_species"}
)

In [72]:
unique_names = df_hosts_long["butterfly_species"].unique()
unique_lower = pd.Series(unique_names).str.lower()
print(len(unique_names), "unique as-is")
print(unique_lower.nunique(), "unique lowercased")

39 unique as-is
39 unique lowercased


In [73]:
print("Total edges:", len(df_hosts_long))
print("Unique plants:", df_hosts_long["species"].nunique())
print("Unique butterflies:", df_hosts_long["butterfly_species"].nunique())

Total edges: 649
Unique plants: 14
Unique butterflies: 39


In [76]:
df_final_to_save = df_final.drop(columns=["file_exists", "file_valid", "num_hosts"], errors="ignore")

In [78]:
df_final_to_save.to_csv("plants_metadata_clean.csv",index=False)

In [79]:
df_hosts_long.to_csv("plant_butterfly_edges.csv", index=False)

In [26]:
df_final_butterflies = pd.read_csv("butterflies_metadata_clean.csv")

In [27]:
plant_side_butterflies = set(df_hosts_long["butterfly_species"].unique())
butterfly_image_species = set(df_final_butterflies["species"].unique())  # your cleaned butterflies df — adjust variable name if different

overlap = plant_side_butterflies & butterfly_image_species
missing = plant_side_butterflies - butterfly_image_species

print(f"Butterflies needed (from plants): {len(plant_side_butterflies)}")
print(f"Butterflies with image data available: {len(overlap)}")
print(f"Butterflies missing image data: {len(missing)}")
print(sorted(missing))

Butterflies needed (from plants): 39
Butterflies with image data available: 1
Butterflies missing image data: 38
['Anthene lycaenina', 'Azanus jesous', 'Azanus ubaldus', 'Azanus uranus', 'Badamia exclamationis', 'Catochrysops strabo', 'Charaxes agrarius', 'Charaxes bharata', 'Colias erate', 'Danaus chrysippus', 'Danaus genutia', 'Delias eucharis', 'Euchrysops cnejus', 'Euploea core', 'Eurema hecabe', 'Everes argiades', 'Freyeria putli', 'Freyeria trochylus', 'Graphium agamemnon', 'Graphium doson', 'Hasora badra', 'Hypolimnas bolina', 'Hypolimnas misippus', 'Jamides celeno', 'Junonia hierta', 'Junonia orithya', 'Lachides parrhasius', 'Lacturnea lacturnus', 'Lampides boeticus', 'Leptotes plinius', 'Luthrodes pandava', 'Neptis hylas', 'Parantica aglea', 'Prosotas dubiosa', 'Prosotas nora', 'Rapala iarbus', 'Tirumala limniace', 'Zizina otis']


In [29]:
df_check = pd.read_csv("../butterflies/data/butterflies/metadata.csv")

for name in ["Charaxes", "Hypolimnas"]:
    matches = df_check[df_check["species"].str.contains(name, case=False, na=False)]
    print(f"\n--- Matches containing '{name}' ---")
    print(matches["species"].unique())


--- Matches containing 'Charaxes' ---
['Charaxes aristogiton' 'Charaxes eudamippus' 'Charaxes moori'
 'Charaxes narcaeus' 'Charaxes schreiber']

--- Matches containing 'Hypolimnas' ---
[]
